# Retail Customer Segmentation — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

UCI Online Retail; raw workbook is downloaded reproducibly by src/data.py.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'retail_customer_segmentation'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from src.data import audit_raw_data, clean_transactions, download_dataset, load_raw_transactions, save_json
from src.evaluation import add_relative_segment_labels, summarize_clusters, validate_segment_output
from src.features import build_customer_features, prepare_clustering_matrix
from src.model import cluster_stability, evaluate_cluster_counts, fit_final_kmeans

PROJECT_DIR = Path(__file__).resolve().parent
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"


def main() -> None:
    raw_dir = DATA_DIR / "raw"
    processed_dir = DATA_DIR / "processed"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

    workbook = download_dataset(raw_dir)
    raw = load_raw_transactions(workbook)
    raw_audit = audit_raw_data(raw)

    clean, cleaning_report = clean_transactions(raw)
    customers = build_customer_features(clean)
    matrix, transformed, _, preprocessing_metadata = prepare_clustering_matrix(customers)

    selection = evaluate_cluster_counts(matrix)
    model, labels = fit_final_kmeans(matrix, selection.selected_k)
    validate_segment_output(customers, labels)

    stability = cluster_stability(matrix, selection.selected_k, labels)
    summary = add_relative_segment_labels(summarize_clusters(customers, labels))

    assignments = customers.copy()
    assignments["cluster"] = labels

    clean.to_csv(processed_dir / "clean_transactions.csv", index=False)
    assignments.to_csv(RESULTS_DIR / "customer_segments.csv", index=False)
    selection.diagnostics.to_csv(RESULTS_DIR / "cluster_diagnostics.csv", index=False)
    summary.to_csv(RESULTS_DIR / "cluster_summary.csv", index=False)

    verification = {
        "verification_pass": True,
        "source": "UCI Machine Learning Repository - Online Retail",
        "raw_audit": raw_audit,
        "cleaning": cleaning_report,
        "customer_feature_rows": int(len(customers)),
        "selected_k": int(selection.selected_k),
        "selected_silhouette": float(
            selection.diagnostics.loc[
                selection.diagnostics["k"].eq(selection.selected_k), "silhouette"
            ].iloc[0]
        ),
        "cluster_stability": stability,
        "preprocessing": preprocessing_metadata,
        "cluster_centres_scaled": model.cluster_centers_.tolist(),
        "output_files": [
            "results/customer_segments.csv",
            "results/cluster_diagnostics.csv",
            "results/cluster_summary.csv",
        ],
        "limitations": [
            "KMeans imposes distance-based partitions and does not prove that natural customer segments exist.",
            "RFM summarizes transaction behaviour and does not capture demographics, channel exposure or profit margin.",
            "Segment names are relative descriptions of this dataset, not universal customer personas.",
            "The analysis is descriptive and should not be interpreted as causal evidence for marketing actions.",
        ],
    }
    save_json(verification, RESULTS_DIR / "verification.json")
    save_json(raw_audit, RESULTS_DIR / "raw_data_audit.json")
    save_json(cleaning_report, RESULTS_DIR / "cleaning_report.json")

    print(json.dumps(verification, indent=2))


if __name__ == "__main__":
    main()


### `src/data.py`


In [ ]:
from __future__ import annotations

import json
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
EXPECTED_COLUMNS = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}


def download_dataset(cache_dir: Path) -> Path:
    """Download the UCI Online Retail workbook and return the local xlsx path."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    archive_path = cache_dir / "online_retail.zip"

    if not archive_path.exists():
        request = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response, archive_path.open("wb") as target:
            for chunk in iter(lambda: response.read(1 << 20), b""):
                target.write(chunk)

    with zipfile.ZipFile(archive_path) as archive:
        workbook_names = [name for name in archive.namelist() if name.lower().endswith(".xlsx")]
        if len(workbook_names) != 1:
            raise ValueError(f"Expected one xlsx workbook, found {workbook_names}")
        workbook_name = workbook_names[0]
        workbook_path = cache_dir / Path(workbook_name).name
        if not workbook_path.exists():
            archive.extract(workbook_name, cache_dir)
            extracted = cache_dir / workbook_name
            if extracted != workbook_path:
                extracted.replace(workbook_path)
    return workbook_path


def load_raw_transactions(workbook_path: Path) -> pd.DataFrame:
    frame = pd.read_excel(workbook_path, engine="openpyxl")
    missing = sorted(EXPECTED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"Source schema is missing expected columns: {missing}")
    return frame


def audit_raw_data(frame: pd.DataFrame) -> dict:
    """Return a machine-readable profile before cleaning."""
    return {
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "missing_by_column": {key: int(value) for key, value in frame.isna().sum().items()},
        "non_positive_quantity_rows": int((pd.to_numeric(frame["Quantity"], errors="coerce") <= 0).sum()),
        "non_positive_unit_price_rows": int((pd.to_numeric(frame["UnitPrice"], errors="coerce") <= 0).sum()),
        "cancelled_invoice_rows": int(frame["InvoiceNo"].astype("string").str.upper().str.startswith("C", na=False).sum()),
        "unique_invoices": int(frame["InvoiceNo"].nunique(dropna=True)),
        "unique_customers": int(frame["CustomerID"].nunique(dropna=True)),
        "date_min": str(pd.to_datetime(frame["InvoiceDate"], errors="coerce").min()),
        "date_max": str(pd.to_datetime(frame["InvoiceDate"], errors="coerce").max()),
    }


def clean_transactions(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Create a customer-level modelling base from messy transactional data.

    Rules are explicit and auditable rather than silently dropping rows:
    - remove exact duplicate lines;
    - require customer, invoice, stock code, timestamp and country;
    - remove cancellations and non-positive quantity/price rows;
    - normalize text and dtypes;
    - create line revenue and validate the final table.
    """
    work = frame.copy()
    initial_rows = len(work)

    duplicate_mask = work.duplicated(keep="first")
    duplicate_rows = int(duplicate_mask.sum())
    work = work.loc[~duplicate_mask].copy()

    for column in ["InvoiceNo", "StockCode", "Description", "Country"]:
        work[column] = work[column].astype("string").str.strip()

    work["InvoiceDate"] = pd.to_datetime(work["InvoiceDate"], errors="coerce")
    work["Quantity"] = pd.to_numeric(work["Quantity"], errors="coerce")
    work["UnitPrice"] = pd.to_numeric(work["UnitPrice"], errors="coerce")
    work["CustomerID"] = pd.to_numeric(work["CustomerID"], errors="coerce")

    work["Country"] = work["Country"].str.replace(r"\s+", " ", regex=True)
    work["Description"] = work["Description"].str.replace(r"\s+", " ", regex=True)
    work["is_cancelled"] = work["InvoiceNo"].str.upper().str.startswith("C", na=False)

    reason_masks = {
        "missing_customer_id": work["CustomerID"].isna(),
        "missing_invoice_no": work["InvoiceNo"].isna() | work["InvoiceNo"].eq(""),
        "missing_stock_code": work["StockCode"].isna() | work["StockCode"].eq(""),
        "missing_invoice_date": work["InvoiceDate"].isna(),
        "missing_country": work["Country"].isna() | work["Country"].eq(""),
        "cancelled_invoice": work["is_cancelled"],
        "non_positive_quantity": work["Quantity"].isna() | work["Quantity"].le(0),
        "non_positive_unit_price": work["UnitPrice"].isna() | work["UnitPrice"].le(0),
    }

    invalid_mask = np.zeros(len(work), dtype=bool)
    removed_by_rule: dict[str, int] = {}
    for name, mask in reason_masks.items():
        removed_by_rule[name] = int(mask.sum())
        invalid_mask |= mask.to_numpy()

    clean = work.loc[~invalid_mask].copy()
    clean["CustomerID"] = clean["CustomerID"].round().astype("int64").astype("string")
    clean["Quantity"] = clean["Quantity"].astype("int64")
    clean["line_revenue"] = clean["Quantity"] * clean["UnitPrice"]

    clean = clean[
        [
            "InvoiceNo",
            "InvoiceDate",
            "CustomerID",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "line_revenue",
            "Country",
        ]
    ].sort_values(["InvoiceDate", "InvoiceNo", "StockCode"], kind="stable")
    clean = clean.reset_index(drop=True)

    if clean.empty:
        raise ValueError("Cleaning removed every row; inspect source or cleaning rules")
    if clean.duplicated().any():
        raise AssertionError("Exact duplicates remain after cleaning")
    if clean[["InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "Quantity", "UnitPrice", "Country"]].isna().any().any():
        raise AssertionError("Required final fields contain missing values")
    if not clean["Quantity"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive quantities")
    if not clean["UnitPrice"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive prices")
    if not clean["line_revenue"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive line revenue")

    report = {
        "source_rows": int(initial_rows),
        "exact_duplicates_removed": duplicate_rows,
        "rule_counts_before_combining": removed_by_rule,
        "rows_removed_after_deduplication": int(len(work) - len(clean)),
        "final_rows": int(len(clean)),
        "retained_share": float(len(clean) / initial_rows),
        "final_unique_customers": int(clean["CustomerID"].nunique()),
        "final_unique_invoices": int(clean["InvoiceNo"].nunique()),
        "final_revenue": float(clean["line_revenue"].sum()),
        "final_date_min": clean["InvoiceDate"].min().isoformat(),
        "final_date_max": clean["InvoiceDate"].max().isoformat(),
    }
    return clean, report


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


### `src/features.py`


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

RFM_COLUMNS = ["recency_days", "frequency_orders", "monetary_value"]


def build_customer_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate clean line-level transactions into one row per customer."""
    required = {
        "CustomerID",
        "InvoiceNo",
        "InvoiceDate",
        "Quantity",
        "line_revenue",
    }
    missing = sorted(required - set(transactions.columns))
    if missing:
        raise ValueError(f"Missing required transaction columns: {missing}")

    snapshot_date = transactions["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

    customer = (
        transactions.groupby("CustomerID", as_index=False)
        .agg(
            last_purchase=("InvoiceDate", "max"),
            first_purchase=("InvoiceDate", "min"),
            frequency_orders=("InvoiceNo", "nunique"),
            monetary_value=("line_revenue", "sum"),
            total_items=("Quantity", "sum"),
            transaction_lines=("InvoiceNo", "size"),
        )
    )

    customer["recency_days"] = (snapshot_date - customer["last_purchase"].dt.normalize()).dt.days
    customer["customer_tenure_days"] = (
        customer["last_purchase"].dt.normalize() - customer["first_purchase"].dt.normalize()
    ).dt.days + 1
    customer["average_order_value"] = customer["monetary_value"] / customer["frequency_orders"]

    if customer["CustomerID"].duplicated().any():
        raise AssertionError("Customer feature table is not one row per customer")
    if customer[RFM_COLUMNS].isna().any().any():
        raise AssertionError("RFM features contain missing values")
    if customer["recency_days"].lt(0).any():
        raise AssertionError("Recency cannot be negative")
    if customer["frequency_orders"].le(0).any():
        raise AssertionError("Frequency must be positive")
    if customer["monetary_value"].le(0).any():
        raise AssertionError("Monetary value must be positive")

    return customer.sort_values("CustomerID").reset_index(drop=True)


def cap_extreme_values(
    frame: pd.DataFrame,
    columns: list[str] | None = None,
    lower_quantile: float = 0.01,
    upper_quantile: float = 0.99,
) -> tuple[pd.DataFrame, dict]:
    """Winsorise extreme RFM values while keeping every customer in the analysis."""
    columns = columns or RFM_COLUMNS
    capped = frame.copy()
    caps: dict[str, dict[str, float]] = {}

    for column in columns:
        lower = float(capped[column].quantile(lower_quantile))
        upper = float(capped[column].quantile(upper_quantile))
        if lower > upper:
            raise ValueError(f"Invalid quantile bounds for {column}")
        capped[column] = capped[column].clip(lower=lower, upper=upper)
        caps[column] = {"lower": lower, "upper": upper}

    return capped, caps


def prepare_clustering_matrix(
    customer_features: pd.DataFrame,
) -> tuple[np.ndarray, pd.DataFrame, RobustScaler, dict]:
    """Log-transform skewed RFM features and robust-scale them for distance models."""
    capped, caps = cap_extreme_values(customer_features)

    transformed = pd.DataFrame(index=capped.index)
    transformed["recency_log1p"] = np.log1p(capped["recency_days"].astype(float))
    transformed["frequency_log1p"] = np.log1p(capped["frequency_orders"].astype(float))
    transformed["monetary_log1p"] = np.log1p(capped["monetary_value"].astype(float))

    scaler = RobustScaler()
    matrix = scaler.fit_transform(transformed)

    if not np.isfinite(matrix).all():
        raise AssertionError("Clustering matrix contains non-finite values")

    metadata = {
        "winsorisation_caps": caps,
        "transforms": {
            "recency_days": "log1p after 1st/99th percentile clipping",
            "frequency_orders": "log1p after 1st/99th percentile clipping",
            "monetary_value": "log1p after 1st/99th percentile clipping",
        },
        "scaler": "RobustScaler",
    }
    return matrix, transformed, scaler, metadata


### `src/model.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

SEED = 42


@dataclass(frozen=True)
class ClusterSelection:
    selected_k: int
    diagnostics: pd.DataFrame


def evaluate_cluster_counts(
    matrix: np.ndarray,
    k_values: range | list[int] = range(2, 11),
    seed: int = SEED,
) -> ClusterSelection:
    """Compare candidate k values with multiple complementary diagnostics."""
    if len(matrix) < 10:
        raise ValueError("Too few customers for meaningful clustering")

    rows: list[dict] = []
    for k in k_values:
        if k >= len(matrix):
            continue
        model = KMeans(n_clusters=k, n_init=50, random_state=seed)
        labels = model.fit_predict(matrix)
        rows.append(
            {
                "k": int(k),
                "silhouette": float(silhouette_score(matrix, labels)),
                "davies_bouldin": float(davies_bouldin_score(matrix, labels)),
                "calinski_harabasz": float(calinski_harabasz_score(matrix, labels)),
                "inertia": float(model.inertia_),
                "smallest_cluster_share": float(pd.Series(labels).value_counts(normalize=True).min()),
            }
        )

    diagnostics = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    if diagnostics.empty:
        raise ValueError("No valid cluster counts were evaluated")

    # Silhouette is the primary selection rule. A minimum cluster-size guard avoids
    # promoting a superficially strong solution dominated by tiny fragments.
    eligible = diagnostics.loc[diagnostics["smallest_cluster_share"] >= 0.02]
    selection_pool = eligible if not eligible.empty else diagnostics
    selected_k = int(selection_pool.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
    return ClusterSelection(selected_k=selected_k, diagnostics=diagnostics)


def fit_final_kmeans(matrix: np.ndarray, n_clusters: int, seed: int = SEED) -> tuple[KMeans, np.ndarray]:
    model = KMeans(n_clusters=n_clusters, n_init=100, random_state=seed)
    labels = model.fit_predict(matrix)
    return model, labels


def cluster_stability(
    matrix: np.ndarray,
    n_clusters: int,
    reference_labels: np.ndarray,
    seeds: tuple[int, ...] = (7, 19, 31, 43, 59, 71, 83, 97),
) -> dict:
    """Estimate sensitivity to KMeans initialization using adjusted Rand index."""
    scores = []
    for seed in seeds:
        model = KMeans(n_clusters=n_clusters, n_init=20, random_state=seed)
        candidate = model.fit_predict(matrix)
        scores.append(float(adjusted_rand_score(reference_labels, candidate)))

    return {
        "metric": "adjusted_rand_index",
        "runs": len(scores),
        "mean": float(np.mean(scores)),
        "min": float(np.min(scores)),
        "max": float(np.max(scores)),
        "scores": scores,
    }


### `src/evaluation.py`


In [ ]:
from __future__ import annotations

import pandas as pd


def summarize_clusters(customer_features: pd.DataFrame, labels) -> pd.DataFrame:
    """Return an interpretable customer-segment table using original business units."""
    segmented = customer_features.copy()
    segmented["cluster"] = labels

    summary = (
        segmented.groupby("cluster", as_index=False)
        .agg(
            customers=("CustomerID", "size"),
            median_recency_days=("recency_days", "median"),
            median_frequency_orders=("frequency_orders", "median"),
            median_monetary_value=("monetary_value", "median"),
            mean_order_value=("average_order_value", "mean"),
            total_revenue=("monetary_value", "sum"),
        )
        .sort_values("median_monetary_value", ascending=False)
        .reset_index(drop=True)
    )
    summary["customer_share"] = summary["customers"] / summary["customers"].sum()
    summary["revenue_share"] = summary["total_revenue"] / summary["total_revenue"].sum()
    return summary


def add_relative_segment_labels(summary: pd.DataFrame) -> pd.DataFrame:
    """Attach descriptive labels based on observed cluster profiles, not hidden assumptions."""
    labelled = summary.copy()
    recency_mid = labelled["median_recency_days"].median()
    frequency_mid = labelled["median_frequency_orders"].median()
    monetary_mid = labelled["median_monetary_value"].median()

    def describe(row) -> str:
        recent = row["median_recency_days"] <= recency_mid
        frequent = row["median_frequency_orders"] >= frequency_mid
        valuable = row["median_monetary_value"] >= monetary_mid
        if recent and frequent and valuable:
            return "high_value_active"
        if recent and not frequent and valuable:
            return "recent_high_spend"
        if recent and not valuable:
            return "recent_lower_value"
        if not recent and valuable:
            return "valuable_at_risk"
        return "inactive_lower_value"

    labelled["relative_profile"] = labelled.apply(describe, axis=1)
    return labelled


def validate_segment_output(customer_features: pd.DataFrame, labels) -> None:
    if len(customer_features) != len(labels):
        raise AssertionError("Every customer must receive exactly one cluster label")
    if pd.Series(labels).isna().any():
        raise AssertionError("Cluster labels contain missing values")
    if pd.Series(labels).nunique() < 2:
        raise AssertionError("Clustering collapsed to fewer than two segments")


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


## Application engineering layer

The original project above is intentionally preserved. The cells below expose additional canonical Python from this same project—pipelines, APIs, feature code, evaluation, tests, monitoring and other application logic—so the notebook works as a single recruiter-facing project while the modular files remain the production source of truth.


### Canonical source: `src/__init__.py`


In [ ]:
"""Retail customer data-cleaning and segmentation project."""


### Canonical source: `tests/test_cleaning.py`


In [ ]:
import pandas as pd

from src.data import audit_raw_data, clean_transactions


def _fixture() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "InvoiceNo": ["10001", "10001", "C10002", "10003", "10004", "10005", "10006"],
            "StockCode": ["A1", "A1", "A2", "A3", "A4", "A5", "A6"],
            "Description": [" Mug ", " Mug ", "Return", "Tea", "Plate", "Bowl", "Glass"],
            "Quantity": [2, 2, -1, 1, 0, 1, 3],
            "InvoiceDate": pd.to_datetime(
                [
                    "2025-01-01 09:00",
                    "2025-01-01 09:00",
                    "2025-01-02 10:00",
                    "2025-01-03 11:00",
                    "2025-01-04 12:00",
                    "2025-01-05 13:00",
                    "2025-01-06 14:00",
                ]
            ),
            "UnitPrice": [5.0, 5.0, 4.0, 3.0, 2.0, 0.0, 7.0],
            "CustomerID": [1, 1, 2, None, 4, 5, 6],
            "Country": [" United Kingdom "] * 7,
        }
    )


def test_raw_audit_counts_quality_problems():
    audit = audit_raw_data(_fixture())
    assert audit["rows"] == 7
    assert audit["exact_duplicate_rows"] == 1
    assert audit["cancelled_invoice_rows"] == 1
    assert audit["non_positive_quantity_rows"] == 2
    assert audit["non_positive_unit_price_rows"] == 1


def test_cleaning_is_explicit_and_produces_valid_rows():
    clean, report = clean_transactions(_fixture())

    assert report["exact_duplicates_removed"] == 1
    assert len(clean) == 2
    assert set(clean["CustomerID"]) == {"1", "6"}
    assert clean["Quantity"].gt(0).all()
    assert clean["UnitPrice"].gt(0).all()
    assert clean["line_revenue"].gt(0).all()
    assert not clean.duplicated().any()
    assert clean.loc[clean["CustomerID"].eq("1"), "Country"].iloc[0] == "United Kingdom"


### Canonical source: `tests/test_features_and_model.py`


In [ ]:
import numpy as np
import pandas as pd

from src.evaluation import summarize_clusters, validate_segment_output
from src.features import build_customer_features, prepare_clustering_matrix
from src.model import cluster_stability, evaluate_cluster_counts, fit_final_kmeans


def _clean_transactions() -> pd.DataFrame:
    rows = []
    start = pd.Timestamp("2025-01-01")
    for customer in range(1, 31):
        orders = 1 + (customer % 5)
        for order in range(orders):
            rows.append(
                {
                    "InvoiceNo": f"{customer:03d}-{order:02d}",
                    "InvoiceDate": start + pd.Timedelta(days=customer * 2 + order),
                    "CustomerID": str(customer),
                    "StockCode": f"S{order:02d}",
                    "Description": "fixture",
                    "Quantity": 1 + customer % 4,
                    "UnitPrice": 2.0 + customer,
                    "line_revenue": float((1 + customer % 4) * (2.0 + customer)),
                    "Country": "United Kingdom",
                }
            )
    return pd.DataFrame(rows)


def test_customer_features_are_one_row_per_customer():
    customer = build_customer_features(_clean_transactions())
    assert len(customer) == 30
    assert customer["CustomerID"].is_unique
    assert customer["recency_days"].ge(0).all()
    assert customer["frequency_orders"].gt(0).all()
    assert customer["monetary_value"].gt(0).all()


def test_clustering_pipeline_returns_stable_valid_shapes():
    customer = build_customer_features(_clean_transactions())
    matrix, transformed, _, metadata = prepare_clustering_matrix(customer)

    assert matrix.shape == (30, 3)
    assert transformed.shape == (30, 3)
    assert np.isfinite(matrix).all()
    assert metadata["scaler"] == "RobustScaler"

    selection = evaluate_cluster_counts(matrix, k_values=[2, 3, 4])
    assert selection.selected_k in {2, 3, 4}
    assert set(selection.diagnostics["k"]) == {2, 3, 4}

    _, labels = fit_final_kmeans(matrix, selection.selected_k)
    validate_segment_output(customer, labels)
    summary = summarize_clusters(customer, labels)
    assert summary["customers"].sum() == 30
    assert np.isclose(summary["customer_share"].sum(), 1.0)

    stability = cluster_stability(matrix, selection.selected_k, labels, seeds=(3, 5, 7))
    assert stability["runs"] == 3
    assert 0.0 <= stability["min"] <= 1.0


## Portfolio depth check

**Meaningful code lines currently visible in this notebook:** 535. The portfolio aims for roughly **1,000 meaningful lines** per major project (normally about 800–1,200), and this notebook is below the preferred band and should gain substantive project-specific depth rather than filler.

Line count is not a quality metric by itself. Additional code should only be added when it strengthens the real application: data validation, cleaning, EDA, feature engineering, modelling, tuning, error analysis, explainability, inference, testing, monitoring, APIs, reproducibility or business logic.
